# 03 - Data Preparation

El objetivo de este notebook es preparar el dataset para las etapas posteriores
de segmentación de clientes.

A partir de los hallazgos identificados durante el análisis exploratorio, se 
abordarán los valores ausentes, las observaciones potencialmente anómalas y las
variables no informativas. Además, se crearán nuevas características que
representan de forma más directa el perfil y comportamiento de los clientes.

## Estructura

1. Carga de datos
2. Tratamiento de valores ausentes
3. Tratamiento de anomalías y valores atípicos
4. Eliminación de variables no informativas
5. Feature engineering
6. Validación del dataset preparado
7. Exportación de datos procesados

## 1. Carga de datos

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
DATA_PATH = Path("../data/raw/marketing_campaign.csv")

df = pd.read_csv(DATA_PATH, sep="\t")

df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")

df.shape

(2240, 29)

## 2. Tratamiento de valores ausentes

Durante el EDA se identificaron 24 valores ausentes en la variable `Income`.
Antes de aplicar una estrategia de imputación, se analiza si estos registros presentan algún patrón relevante respecto al resto de clientes.

In [3]:
df.isna().sum()[df.isna().sum() > 0]

Income    24
dtype: int64

In [4]:
missing_income = df[df["Income"].isna()]

missing_income[
    [
        "ID",
        "Year_Birth",
        "Education",
        "Marital_Status",
        "Kidhome",
        "Teenhome",
        "Dt_Customer",
        "Recency",
    ]
]

,ID,Year_Birth,Education,Marital_Status,Kidhome,Teenhome,Dt_Customer,Recency
10,1994,1983,Graduation,Married,1,0,2013-11-15,11
27,5255,1986,Graduation,Single,1,0,2013-02-20,19
43,7281,1959,PhD,Single,0,0,2013-11-05,80
48,7244,1951,Graduation,Single,2,1,2014-01-01,96
58,8557,1982,Graduation,Single,1,0,2013-06-17,57
71,10629,1973,2n Cycle,Married,1,0,2012-09-14,25
90,8996,1957,PhD,Married,2,1,2012-11-19,4
91,9235,1957,Graduation,Single,1,1,2014-05-27,45
92,5798,1973,Master,Together,0,0,2013-11-23,87
128,8268,1961,PhD,Married,0,1,2013-07-11,23


In [5]:
missing_income["Education"].value_counts()

Education
Graduation    11
PhD            5
Master         5
2n Cycle       3
Name: count, dtype: int64

In [6]:
missing_income["Marital_Status"].value_counts()

Marital_Status
Single      9
Married     7
Together    7
Widow       1
Name: count, dtype: int64

In [7]:
df.groupby("Education")["Income"].agg(["count", "median", "mean"]).sort_values("median")

,count,median,mean
Education,,,
Basic,54,20744.0,20306.259259
2n Cycle,200,46805.0,47633.190000
Master,365,50943.0,52917.534247
Graduation,1116,52028.5,52720.373656
PhD,481,55212.0,56145.313929


### 2.1. Estrategia de imputación

Los valores ausentes de `Income` representan aproximadamente el 1,1 % de los
registros, por lo que se decide conservar estas observaciones e imputar sus
ingresos.

Las medianas de `Income` presentan diferencias relevantes entre los distintos
niveles educativos. Por este motivo, se utiliza la mediana de ingresos
correspondiente a cada categoría de `Education`.

Se utiliza la mediana en lugar de la media debido a la presencia de valores
extremos en la distribución de `Income`.

In [8]:
income_median_by_education = df.groupby("Education")["Income"].transform("median")

df["Income"] = df["Income"].fillna(income_median_by_education)

In [9]:
# Comprobamos que la imputación se haya realizado
df["Income"].isna().sum()

np.int64(0)

## 3. Tratamiento de anomalías y valores atípicos

Durante el EDA se identificaron observaciones potencialmente anómalas en
`Year_Birth`, `Income` y algunas categorías de `Marital_Status`.

Estas observaciones se evalúan individualmente antes de decidir su tratamiento,
evitando modificar o eliminar datos únicamente por presentar valores poco frecuentes o estadísticamente atípicos.

### 3.1 Año de nacimiento

Durante el EDA se identificaron tres valores especialmente bajos en
`Year_Birth`: 1893, 1899 y 1900, mientras que el siguiente valor observado es 1940.

Se inspeccionan estos registros para comprobar si existen otras anomalías
evidentes que justifiquen la eliminación completa de los clientes.

In [10]:
birth_outliers = df[df["Year_Birth"] < 1940]

birth_outliers.T

,192,239,339
ID,7829,11004,1150
Year_Birth,1900,1893,1899
Education,2n Cycle,2n Cycle,PhD
Marital_Status,Divorced,Single,Together
Income,36640.0,60182.0,83532.0
Kidhome,1,0,0
Teenhome,0,1,0
Dt_Customer,2013-09-26 00:00:00,2014-05-17 00:00:00,2013-09-26 00:00:00
Recency,99,23,36
MntWines,15,8,755


La inspección de estos registros no muestra anomalías evidentes en el resto
de sus variables. Dado que no existe información suficiente para determinar
que los años de nacimiento sean incorrectos ni para inferir valores
alternativos, se mantienen los datos originales.

Su posible impacto se tendrá en cuenta posteriormente al construir `Age` y
seleccionar las variables utilizadas para la segmentación.

### 3.2 Valores extremos en Income

Durante el EDA se identificaron varios valores elevados en `Income`.
Se inspeccionan los mayores ingresos para evaluar especialmente el valor
máximo observado.

In [11]:
df.nlargest(10, "Income")[
    [
        "ID",
        "Income",
        "Education",
        "Marital_Status",
        "MntWines",
        "MntMeatProducts",
        "NumWebPurchases",
        "NumCatalogPurchases",
        "NumStorePurchases",
    ]
]

,ID,Income,Education,Marital_Status,MntWines,MntMeatProducts,NumWebPurchases,NumCatalogPurchases,NumStorePurchases
2233,9432,666666.0,Graduation,Together,9,18,3,1,3
617,1503,162397.0,PhD,Together,85,16,0,0,1
687,1501,160803.0,PhD,Married,55,1622,0,28,1
1300,5336,157733.0,Master,Together,39,9,1,0,1
164,8475,157243.0,PhD,Married,20,1582,0,22,0
1653,4931,157146.0,Graduation,Together,1,1725,0,28,0
2132,11181,156924.0,PhD,Married,2,2,0,0,0
655,5555,153924.0,Graduation,Divorced,1,1,0,0,0
1898,4619,113734.0,PhD,Single,6,3,27,0,0
646,4611,105471.0,Graduation,Together,1009,104,9,8,13


El valor máximo de `Income` es 666666, muy superior al resto de valoresobservados.

Sin embargo, la inspección del registro no permite determinar con certeza que se trate de un error. Por este motivo, se conserva el valor original y no se eliminan observaciones únicamente por su condición de outlier.

Dado que los algoritmos basados en distancias como K-Means son sensibles a valores extremos, su impacto se evaluará posteriormente durante la selección y transformación de las variables utilizadas para el clustering.

### 3.3 Estado civil

Se analiza la distribución de `Marital_Status` para revisar las categorías con una frecuencia especialmente reducida.

In [12]:
df["Marital_Status"].value_counts()

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

In [13]:
rare_status = ["Alone", "Absurd", "YOLO"]

df[df["Marital_Status"].isin(rare_status)][
    ["ID", "Year_Birth", "Education", "Marital_Status", "Income", "Kidhome", "Teenhome"]
]

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome
131,433,1958,Master,Alone,61331.0,1,1
138,7660,1973,PhD,Alone,35860.0,1,1
153,92,1988,Graduation,Alone,34176.0,1,0
2093,7734,1993,Graduation,Absurd,79244.0,0,0
2134,4369,1957,Master,Absurd,65487.0,0,0
2177,492,1973,PhD,YOLO,48432.0,0,1
2202,11133,1973,PhD,YOLO,48432.0,0,1


`Alone`, `Absurd` y `YOLO` presentan una frecuencia muy reducida, con un total de siete observaciones.

Aunque estas categorías podrían agruparse para reducir la cardinalidad, no
existe una necesidad analítica que justifique hacerlo en esta fase. Además,
agrupar estados como `Single`, `Divorced` y `Widow` supondría perder
información potencialmente útil para el perfilado de clientes.

Por tanto, se conserva `Marital_Status` sin modificaciones. La necesidad de
agrupar o codificar sus categorías se reevaluará posteriormente en función
de las variables seleccionadas para la segmentación.

## 4. Eliminación de variables no informativas

Durante el análisis exploratorio se identificaron variables constantes.

Al presentar el mismo valor para todos los clientes, estas variables no aportan capacidad discriminativa para el análisis ni para la posterior segmentación.

In [14]:
constant_cols = [col for col in df.columns if df[col].nunique() == 1]

constant_cols

['Z_CostContact', 'Z_Revenue']

In [15]:
df = df.drop(columns=constant_cols)


In [16]:
df.shape

(2240, 27)

`Z_CostContact` y `Z_Revenue` son eliminadas al presentar un único valor en todo el dataset.

`ID`, en cambio, se conserva para mantener la trazabilidad de los clientes, aunque no se utilizará posteriormente como variable de entrada para el clustering.

## 5. Feature Engineering

A partir de las variables originales se crean nuevas características que representan de forma más directa el perfil y comportamiento de compra de los clientes.

Las variables derivadas se utilizarán posteriormente para el análisis RFM, el clustering y el perfilado de los segmentos.

### 5.1 Referencia temporal

El dataset no proporciona una fecha explícita de extracción o referencia.
Para mantener coherencia temporal con los propios datos, se analiza el rango de fechas de incorporación de los clientes.

In [17]:
df["Dt_Customer"].agg(["min", "max"])

min   2012-07-30
max   2014-06-29
Name: Dt_Customer, dtype: datetime64[us]

In [18]:
df["Dt_Customer"].dt.year.value_counts().sort_index()

Dt_Customer
2012     494
2013    1189
2014     557
Name: count, dtype: int64

In [19]:
reference_date = df["Dt_Customer"].max()
reference_date

Timestamp('2014-06-29 00:00:00')

La fecha de alta más reciente observada es `2014-06-29`. Esta fecha se utiliza como referencia temporal interna para construir variables relacionadas con la edad y la antigüedad de los clientes.

Esta elección constituye una aproximación basada en la información disponible, ya que el dataset no proporciona una fecha explícita de extracción.

### 5.2 Edad del cliente

`Year_Birth` resulta menos interpretable directamente que la edad del cliente.
Se crea `Age` utilizando 2014, año correspondiente a la fecha de referencia establecida anteriormente.

La variable representa, por tanto, una edad aproximada durante el periodo del dataset y no la edad actual del cliente.

In [20]:
df["Age"] = reference_date.year - df["Year_Birth"]

In [21]:
df["Age"].describe()

count    2240.000000
mean       45.194196
std        11.984069
min        18.000000
25%        37.000000
50%        44.000000
75%        55.000000
max       121.000000
Name: Age, dtype: float64

In [22]:
df.nlargest(10, "Age")[["ID", "Year_Birth", "Age"]]

,ID,Year_Birth,Age
239,11004,1893,121
339,1150,1899,115
192,7829,1900,114
1950,6663,1940,74
424,6932,1941,73
39,2968,1943,71
358,6142,1943,71
415,7106,1943,71
894,8800,1943,71
1150,1453,1943,71


Los tres valores extremos identificados previamente en `Year_Birth` generan también valores extremos en `Age`.

Se mantienen en el dataset preparado para preservar la información original, pero su inclusión como variable de clustering se evaluará posteriormente debido a la sensibilidad de K-Means a valores extremos.

### 5.3 Antigüedad como cliente

Se crea `CustomerTenure` como el número de días transcurridos entre la fecha de alta de cada cliente y la fecha de referencia (`2014-06-29`).

Esta variable permite representar cuantitativamente la antigüedad relativa de cada cliente dentro del periodo cubierto por los datos.


In [23]:
df["CustomerTenure"] = (reference_date - df["Dt_Customer"]).dt.days

In [24]:
df["CustomerTenure"].describe()

count    2240.000000
mean      353.582143
std       202.122512
min         0.000000
25%       180.750000
50%       355.500000
75%       529.000000
max       699.000000
Name: CustomerTenure, dtype: float64

### 5.4 Hijos en el hogar

`Kidhome` y `Teenhome` representan respectivamente el número de niños y adolescentes presentes en el hogar. Se crea `Children` para disponer también de una medida agregada del número total de hijos en el hogar.

In [25]:
df["Children"] = df["Kidhome"] + df["Teenhome"]

In [26]:
df["Children"].value_counts().sort_index()

Children
0     638
1    1128
2     421
3      53
Name: count, dtype: int64

### 5.5 Gasto total

Se crea `TotalSpend` como la suma del gasto realizado en todas las categorías de producto disponibles en el dataset.

In [27]:
spending_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds",
]

df["TotalSpend"] = df[spending_cols].sum(axis=1)

In [28]:
df["TotalSpend"].describe()

count    2240.000000
mean      605.798214
std       602.249288
min         5.000000
25%        68.750000
50%       396.000000
75%      1045.500000
max      2525.000000
Name: TotalSpend, dtype: float64

### 5.6 Compras totales

Se crea `TotalPurchases` como el número total de compras realizadas mediante los tres canales disponibles: web, catálogo y tienda.

`NumDealsPurchases` no se incluye en esta suma, ya que representa compras realizadas con descuento y no un canal independiente. Incluirla podría provocar doble contabilización.

In [29]:
purchase_cols = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]

df["TotalPurchases"] = df[purchase_cols].sum(axis=1)

In [30]:
df["TotalPurchases"].describe()

count    2240.000000
mean       12.537054
std         7.205741
min         0.000000
25%         6.000000
50%        12.000000
75%        18.000000
max        32.000000
Name: TotalPurchases, dtype: float64

### 5.7 Aceptaciones de campañas anteriores

Las variables `AcceptedCmp1` a `AcceptedCmp5` indican si el cliente aceptó cada una de las cinco campañas de marketing anteriores.

Se crea `CampaignAcceptances` como el número total de campañas anteriores aceptadas por cada cliente. Esta variable permite resumir el historial de respuesta del cliente a las acciones de marketing.

`Response` no se incluye en esta suma, ya que representa la respuesta a la campaña más reciente. Se mantiene como variable independiente para poder analizar posteriormente su relación con los segmentos obtenidos.

In [31]:
campaign_cols = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5",
]

df["CampaignAcceptances"] = df[campaign_cols].sum(axis=1)

In [32]:
df["CampaignAcceptances"].value_counts().sort_index()

CampaignAcceptances
0    1777
1     325
2      83
3      44
4      11
Name: count, dtype: int64

La mayoría de los clientes (1777 de 2240, aproximadamente el 79,3 %) no aceptó ninguna de las cinco campañas anteriores.

A medida que aumenta el número de campañas aceptadas, disminuye notablemente el número de clientes. El máximo observado es de cuatro campañas aceptadas, sin clientes que hayan aceptado las cinco.

Esta variable permitirá resumir el historial de respuesta a campañas de cada cliente y podrá utilizarse posteriormente para caracterizar los segmentos obtenidos.

### 5.8 Resumen de variables derivadas

In [33]:
derived_cols = [
    "Age",
    "CustomerTenure",
    "Children",
    "TotalSpend",
    "TotalPurchases",
    "CampaignAcceptances",
]

df[derived_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,2240.0,45.194196,11.984069,18.0,37.00,44.0,55.0,121.0
CustomerTenure,2240.0,353.582143,202.122512,0.0,180.75,355.5,529.0,699.0
Children,2240.0,0.950446,0.751803,0.0,0.00,1.0,1.0,3.0
TotalSpend,2240.0,605.798214,602.249288,5.0,68.75,396.0,1045.5,2525.0
TotalPurchases,2240.0,12.537054,7.205741,0.0,6.00,12.0,18.0,32.0
CampaignAcceptances,2240.0,0.297768,0.678381,0.0,0.00,0.0,0.0,4.0


Las variables derivadas presentan valores completos para los 2240 clientes.

`Age` conserva los valores extremos derivados de los años de nacimiento identificados previamente. Estos valores no se modifican en esta fase, perosu impacto deberá evaluarse antes de utilizar la variable en algoritmos sensibles a valores extremos como K-Means.

`CustomerTenure`, `Children`, `TotalSpend`, `TotalPurchases` y `CampaignAcceptances` resumen diferentes dimensiones del perfil y comportamiento de los clientes.

La selección definitiva de variables para la segmentación se realizará posteriormente, evitando introducir variables redundantes o características que puedan distorsionar las distancias utilizadas por K-Means.

## 6. Validación del dataset preparado

Una vez completadas las transformaciones, se realiza una validación final del dataset para comprobar su estructura, la presencia de valores ausentes o duplicados, la unicidad del identificador y la coherencia básica de lasvariables derivadas.

### 6.1 Estructura final

In [34]:
df.shape

(2240, 33)

### 6.2 Valores ausentes

In [35]:
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

### 6.3 Duplicados e identificadores

In [36]:
df.duplicated().sum()

np.int64(0)

In [39]:
df["ID"].nunique(), len(df)

(2240, 2240)

### 6.4 Consistencia de variables derivadas

Finalmente, se comprueban los valores mínimos y máximos de las variables creadas durante el feature engineering para detectar posibles resultados incompatibles con sus definiciones.

In [38]:
df[derived_cols].agg(["min", "max"]).T

,min,max
Age,18,121
CustomerTenure,0,699
Children,0,3
TotalSpend,5,2525
TotalPurchases,0,32
CampaignAcceptances,0,4


## 7. Exportación de datos procesados

Una vez finalizada la preparación y validación de los datos, se exporta el dataset resultante para utilizarlo como punto de partida en las siguientes etapas del proyecto.

De esta forma, los análisis posteriores pueden trabajar sobre una versión consistente de los datos sin repetir las transformaciones realizadas en este notebook.

In [ ]:
OUTPUT_PATH = Path("../data/processed/customer_segmentation_prepared.csv")

df.to_csv(OUTPUT_PATH, index=False)

OUTPUT_PATH

PosixPath('../data/processed/customer_segmentation_prepared.csv')

In [ ]:
prepared_df = pd.read_csv(OUTPUT_PATH, parse_dates=["Dt_Customer"])

prepared_df.shape

(2240, 33)

El dataset procesado se ha exportado correctamente con 2240 registros y 33 variables. Este archivo constituye la versión preparada de los datos y se utilizará como punto de partida para las siguientes etapas de segmentación de clientes.